In [3]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print("*********************-----------------------------------------libraries imported successfully!!!!!!------------------------****************")
print(pd.__version__)
print(requests.__version__)
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(np.__version__)


*********************-----------------------------------------libraries imported successfully!!!!!!------------------------****************
2.2.2
2.32.4
2026-05-27 09:09:55
2.0.2


In [5]:
raw_df=pd.read_csv('messy_sales_data.csv')
print(raw_df.shape[0])
print(raw_df.shape[1])
print(raw_df.columns.tolist())
raw_df.head()

30
9
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [6]:
print("missing values per column")
print(raw_df.isnull().sum())
print("data types")
print(raw_df.dtypes)
print("duplicates")
print(raw_df.duplicated().sum())
print("unique categories are:",raw_df['category'].dropna().unique().tolist())
print("sample order date:",raw_df['order_date'].dropna().unique()[:8].tolist())
print("sample name:",raw_df['customer_name'].dropna().unique()[:8].tolist())

missing values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
data types
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
duplicates
0
unique categories are: ['Electronics', 'Accessories']
sample order date: ['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
sample name: ['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer']


In [7]:
df=raw_df.copy()
print("working copy of the data with shape:",{df.shape})

working copy of the data with shape: {(30, 9)}


In [8]:
df=pd.read_csv("messy_sales_data.csv")

print(df.isnull().sum())
print("Total null values:",df.isnull().sum().sum())

order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Total null values: 7


In [9]:
print("Total null values before fixing")
print(df.isnull().sum())
print("Total null values:",df.isnull().sum().sum())
df['customer_name'].fillna('unknown customer',inplace=True)
df['product'].fillna('unknown product',inplace=True)
df['category'].fillna('uncategorised',inplace=True)
median_qty=df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print("\nMissing values filled successfully")
print("Median quantity used:",median_qty)
print("\nTotal null values after fixing")
print(df.isnull().sum())
print("Total null values:",df.isnull().sum().sum())

Total null values before fixing
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Total null values: 7

Missing values filled successfully
Median quantity used: 2.0

Total null values after fixing
order_id         0
customer_name    0
product          0
category         0
quantity         0
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Total null values: 0


In [10]:
print("duplicates")
print(df.duplicated().sum())

duplicates
0


In [11]:
print("rows before duplication:",len(df))
print("duplicates not founf",df.duplicated().sum())
duplicate_mask=df.duplicated(keep=False)
print("duplicates rows")
print(df[duplicate_mask])
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)
print("rows after duplication:",len(df))
print("rows removed",len(raw_df)-len(df))

rows before duplication: 30
duplicates not founf 0
duplicates rows
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep]
Index: []
rows after duplication: 30
rows removed 0


In [13]:
print("sample dates before parsing")
print(df['order_date'].head(10).tolist())

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)

nat_mask = df['order_date'].isnull()

df.loc[nat_mask, 'order_date'] = pd.to_datetime(
    raw_df.loc[nat_mask, 'order_date'],
    dayfirst=True,
    errors='coerce'
)

nat_count = df['order_date'].isnull().sum()

print("unparsable dates remaining", nat_count)

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%B')
df['day_name'] = df['order_date'].dt.strftime('%A')

print("sample dates after parsing")

print(
    df[['year','month','month_name','day_name']]
    .head(10)
)

sample dates before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']
unparsable dates remaining 0
sample dates after parsing
   year  month month_name   day_name
0  2024      1    January     Friday
1  2024      1    January     Sunday
2  2024      1    January     Monday
3  2024      1    January  Wednesday
4  2024      1    January     Friday
5  2024      1    January     Sunday
6  2024      1    January     Friday
7  2024      1    January   Saturday
8  2024      1    January     Monday
9  2024      1    January     Monday


In [14]:
print("names before standarisation")
print(df['customer_name'].tolist())
df['customer_name']=df['customer_name'].str.strip().str.title()
print("names after standarisation")
print(df['customer_name'].unique().tolist())

names before standarisation
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'Ramesh Kumar', 'kiran mehta', 'Deepak Singh', 'unknown customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'SURESH RAO', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Kiran Mehta', 'Deepak Singh', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'ANANYA DAS', 'Preeti Saxena', 'Amit Bose', 'unknown customer', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']
names after standarisation
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknown Customer', 'Ananya Das', 'Vikram Iyer', 'Pooja Gupta', 'Suresh Rao', 'Meera Joshi', 'Arjun Nair', 'Tanvi Mehta', 'Rohit Verma', 'Sneha Reddy', 'Gaurav Shukla', 'Nisha Kapoor', 'Ajay Tiwari', 'Preeti Saxena', 'Amit Bose', 'Rekha Nair', 'Harish Pillai', 'Sanjay Dubey', 'Kavya Nambiar']


In [15]:
wrong_mask=(df['product']=='Keyboard')&(df['category']=='Electronics')
print("rows to fix",wrong_mask.sum())
print("before fix")
print(df.loc[wrong_mask,['product','category']])
df.loc[wrong_mask,'category']='Accesories'
print("after fix")
print(df.loc[wrong_mask,['product','category']])



rows to fix 1
before fix
     product     category
23  Keyboard  Electronics
after fix
     product    category
23  Keyboard  Accesories


In [16]:
print(df['category'].iloc[0:30])

0       Electronics
1       Electronics
2       Accessories
3       Electronics
4       Electronics
5       Accessories
6       Electronics
7       Accessories
8       Electronics
9       Accessories
10      Electronics
11      Accessories
12      Electronics
13      Electronics
14      Accessories
15      Accessories
16      Accessories
17      Electronics
18      Electronics
19      Accessories
20      Electronics
21      Accessories
22      Electronics
23       Accesories
24      Accessories
25      Electronics
26    uncategorised
27      Electronics
28      Accessories
29      Accessories
Name: category, dtype: object


In [17]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print("revenue column created successfully!!!")
print("REVENUE=QUANTITY * UNIT PRICE")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(8).to_string(index=False))


revenue column created successfully!!!
REVENUE=QUANTITY * UNIT PRICE
   customer_name         product  quantity  unit_price  revenue
    Ramesh Kumar          Laptop         2       45000    90000
      Priya Nair unknown product         1       15000    15000
      Amit Verma        Keyboard         3        1200     3600
    Sunita Patel         Monitor         2       22000    44000
    Ramesh Kumar          Laptop         2       45000    90000
     Kiran Mehta           Mouse        10         800     8000
    Deepak Singh      Headphones         2        3500     7000
Unknown Customer          Webcam         1        2500     2500


In [18]:
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")


  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 0
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 14
  DATA QUALITY SCORE : 100/100
  DATA IS CLEAN      : True

  All checks passed. Data is ready for analysis.


In [19]:
output_filename = 'clean_sales_data.csv'
df.to_csv(output_filename, index=False)
print(f"Cleaned data saved to: {output_filename}")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nETL Pipeline for Sales Data: COMPLETE")
print("  EXTRACT   → messy_sales_data.csv loaded")
print("  TRANSFORM → 6 issues fixed (nulls, dupes, dates, names, categories, types)")
print("  LOAD      → clean_sales_data.csv saved")

Cleaned data saved to: clean_sales_data.csv
Final dataset: 30 rows x 14 columns

ETL Pipeline for Sales Data: COMPLETE
  EXTRACT   → messy_sales_data.csv loaded
  TRANSFORM → 6 issues fixed (nulls, dupes, dates, names, categories, types)
  LOAD      → clean_sales_data.csv saved


In [34]:
SERP_API_KEY='YOUR_SERPAPI_KEY_HERE'
SERP_URL='https://serpapi.com/search.json'
SEARCH_QUERY='Data Engineer India'
print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"Search query :{SEARCH_QUERY}")

SerpAPI Key  : Not set (fallback data will be used)
Search query :Data Engineer India


In [38]:
# ============================================================
# CELL 22 — EXTRACT: Fetch Job Listings from SerpAPI
# ============================================================

def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.

    Parameters:
        query     (str): Job search query
        api_key   (str): SerpAPI key
        num_pages (int): Number of pages to fetch

    Returns:
        list: List of job dictionaries
    """

    all_jobs = []

    for page in range(num_pages):
        params = {
            'engine': 'google_jobs',
            'q': query,
            'api_key': api_key,
            'hl': 'en',
            'start': page * 10
        }

        try:
            response = requests.get(SERP_URL, params=params, timeout=15)

            if response.status_code == 200:
                data = response.json()
                jobs = data.get('jobs_results', [])

                for job in jobs:
                    all_jobs.append({
                        'title': job.get('title', 'Unknown Title'),
                        'company': job.get('company_name', 'Unknown Company'),
                        'location': job.get('location', 'Unknown Location'),
                        'posted': job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary': job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type': job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]
                    })

                print(f"Page {page + 1}: fetched {len(jobs)} jobs")

            else:
                print(f"Page {page + 1}: API error {response.status_code}")

        except Exception as e:
            print(f"Page {page + 1}: error — {e}")

    return all_jobs


# ============================================================
# CALL API OR LOAD FALLBACK DATA
# ============================================================

job_records = []

if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")


# ============================================================
# FALLBACK DATA
# ============================================================

if len(job_records) == 0:
    print("Loading realistic fallback job data...")

    job_records = [
        {'title': 'Data Engineer', 'company': 'Infosys', 'location': 'Bangalore, India', 'posted': '3 days ago', 'salary': 'Rs 12-18 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'TCS', 'location': 'Chennai, India', 'posted': '1 week ago', 'salary': 'Not Disclosed', 'job_type': 'Full-time'},
        {'title': 'Senior Data Engineer', 'company': 'Wipro', 'location': 'Pune, India', 'posted': '2 days ago', 'salary': 'Rs 20-28 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Amazon', 'location': 'Hyderabad, India', 'posted': '5 days ago', 'salary': 'Rs 25-35 LPA', 'job_type': 'Full-time'},
        {'title': 'Junior Data Engineer', 'company': 'HCL', 'location': 'Noida, India', 'posted': '4 days ago', 'salary': 'Rs 6-10 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Google', 'location': 'Bangalore, India', 'posted': '1 day ago', 'salary': 'Rs 40-60 LPA', 'job_type': 'Full-time'},
        {'title': 'ETL Developer', 'company': 'Cognizant', 'location': 'Hyderabad, India', 'posted': '6 days ago', 'salary': 'Rs 8-14 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Microsoft', 'location': 'Hyderabad, India', 'posted': '2 days ago', 'salary': 'Rs 30-45 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Pipeline Engineer', 'company': 'Flipkart', 'location': 'Bangalore, India', 'posted': '1 week ago', 'salary': 'Rs 22-30 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Accenture', 'location': 'Mumbai, India', 'posted': '3 days ago', 'salary': 'Rs 10-16 LPA', 'job_type': 'Full-time'},
        {'title': 'Senior Data Engineer', 'company': 'Amazon', 'location': 'Chennai, India', 'posted': '8 days ago', 'salary': 'Rs 28-40 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Infosys', 'location': 'Pune, India', 'posted': '2 weeks ago', 'salary': 'Not Disclosed', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Swiggy', 'location': 'Bangalore, India', 'posted': '4 days ago', 'salary': 'Rs 18-25 LPA', 'job_type': 'Full-time'},
        {'title': 'ETL Developer', 'company': 'Tech Mahindra', 'location': 'Noida, India', 'posted': '5 days ago', 'salary': 'Rs 7-12 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Zomato', 'location': 'Delhi, India', 'posted': '3 days ago', 'salary': 'Rs 15-22 LPA', 'job_type': 'Full-time'},
        {'title': 'Senior Data Engineer', 'company': 'Google', 'location': 'Bangalore, India', 'posted': '1 day ago', 'salary': 'Rs 50-70 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'PayPal', 'location': 'Chennai, India', 'posted': '2 days ago', 'salary': 'Rs 20-32 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Capgemini', 'location': 'Mumbai, India', 'posted': '1 week ago', 'salary': 'Not Disclosed', 'job_type': 'Full-time'},
        {'title': 'Junior Data Engineer', 'company': 'Wipro', 'location': 'Hyderabad, India', 'posted': '3 days ago', 'salary': 'Rs 5-9 LPA', 'job_type': 'Full-time'},
        {'title': 'Data Engineer', 'company': 'Oracle', 'location': 'Bangalore, India', 'posted': '4 days ago', 'salary': 'Rs 18-28 LPA', 'job_type': 'Full-time'}
    ]

    print(f"Fallback data loaded: {len(job_records)} job listings")

else:
    print(f"Using live SerpAPI data: {len(job_records)} jobs")

No SerpAPI key provided — fallback job data will be loaded next.
Loading realistic fallback job data...
Fallback data loaded: 20 job listings


In [45]:
import pandas as pd
df = pd.DataFrame(job_records)
df.to_excel("job_listings.xlsx", index=False)
print("Excel file created successfully!")

Excel file created successfully!


In [44]:
from google.colab import files
files.download("job_listings.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>